# Notebook 02: Baseline Model Training

Trains 3 model types x 3 prediction tasks = 9 models on W1.
Evaluates performance degradation across W2-W5.


In [ ]:
import sys, os, pickle
if 'google.colab' in sys.modules:
    sys.path.insert(0, '/content/FairDrift-code')
else:
    sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

from src import config

# Load checkpoint from Notebook 01
with open('../outputs/checkpoint_01.pkl', 'rb') as f:
    ckpt = pickle.load(f)
df = ckpt['df']
feature_cols = ckpt['feature_cols']
windows = ckpt['windows']
print(f'Loaded: {len(feature_cols)} features, {len(windows)} windows')


## 1. Prepare Training Data (W1)

In [ ]:
# Training on W1 only
W1 = windows[0]
X_train = W1[feature_cols].values
print(f'Training set: {X_train.shape[0]:,} samples, {X_train.shape[1]} features')

# Scale features for LR and MLP
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)


## 2. Train Models (3 types x 3 tasks)

In [ ]:
models = {}  # key: (model_type, task) -> fitted model
results = []  # Performance records

for task in config.TASKS:
    y_train = W1[task].values
    print(f'\n{"="*60}')
    print(f'TASK: {config.TASKS[task]["description"]} (base rate: {y_train.mean():.3f})')
    print(f'{"="*60}')

    # --- Logistic Regression ---
    print('  Training Logistic Regression...')
    lr = LogisticRegression(max_iter=1000, random_state=config.PRIMARY_SEED)
    lr_cv = GridSearchCV(lr, {'C': config.HYPERPARAM_SEARCH['logistic_regression']['C']},
                         cv=5, scoring='roc_auc', n_jobs=-1)
    lr_cv.fit(X_train_scaled, y_train)
    models[('logistic_regression', task)] = lr_cv.best_estimator_
    print(f'    Best C={lr_cv.best_params_["C"]}, CV AUROC={lr_cv.best_score_:.4f}')

    # --- XGBoost ---
    print('  Training XGBoost...')
    xgb_params = config.HYPERPARAM_SEARCH['xgboost']
    xgb_model = xgb.XGBClassifier(
        eval_metric='logloss', random_state=config.PRIMARY_SEED, use_label_encoder=False
    )
    xgb_cv = GridSearchCV(xgb_model, {
        'max_depth': xgb_params['max_depth'],
        'n_estimators': xgb_params['n_estimators'],
        'learning_rate': xgb_params['learning_rate'],
    }, cv=5, scoring='roc_auc', n_jobs=-1)
    xgb_cv.fit(X_train, y_train)  # XGBoost doesn't need scaling
    models[('xgboost', task)] = xgb_cv.best_estimator_
    print(f'    Best params: {xgb_cv.best_params_}, CV AUROC={xgb_cv.best_score_:.4f}')

    # --- MLP ---
    print('  Training MLP...')
    mlp = MLPClassifier(
        hidden_layer_sizes=config.HYPERPARAM_SEARCH['mlp']['hidden_layer_sizes'],
        early_stopping=True,
        n_iter_no_change=config.HYPERPARAM_SEARCH['mlp']['n_iter_no_change'],
        max_iter=config.HYPERPARAM_SEARCH['mlp']['max_iter'],
        random_state=config.PRIMARY_SEED,
        validation_fraction=0.15,
    )
    mlp.fit(X_train_scaled, y_train)
    models[('mlp', task)] = mlp
    # Estimate CV score
    mlp_proba = mlp.predict_proba(X_train_scaled)[:, 1]
    print(f'    Train AUROC={roc_auc_score(y_train, mlp_proba):.4f} (train, not CV)')


## 3. Evaluate Across All Windows (W1-W5)

In [ ]:
print('MODEL PERFORMANCE ACROSS TEMPORAL WINDOWS')
print('=' * 80)

perf_records = []

for (model_type, task), model in models.items():
    for w_idx, window in enumerate(windows):
        X_test = window[feature_cols].values
        y_test = window[task].values

        # Scale if needed
        if model_type in ['logistic_regression', 'mlp']:
            X_eval = scaler.transform(X_test)
        else:
            X_eval = X_test

        y_proba = model.predict_proba(X_eval)[:, 1]
        auroc = roc_auc_score(y_test, y_proba)
        auprc = average_precision_score(y_test, y_proba)
        brier = brier_score_loss(y_test, y_proba)

        perf_records.append({
            'model': model_type,
            'task': task,
            'window': w_idx + 1,
            'auroc': auroc,
            'auprc': auprc,
            'brier': brier,
        })

perf_df = pd.DataFrame(perf_records)

# Summary table
for task in config.TASKS:
    print(f'\n--- {config.TASKS[task]["description"]} ---')
    subset = perf_df[perf_df['task'] == task].pivot(
        index='model', columns='window', values='auroc'
    )
    print(subset.round(4))


In [ ]:
# Figure: AUROC degradation across windows
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for idx, task in enumerate(config.TASKS):
    ax = axes[idx]
    subset = perf_df[perf_df['task'] == task]
    for model_type in config.MODEL_TYPES:
        data = subset[subset['model'] == model_type]
        ax.plot(data['window'], data['auroc'], 'o-', label=model_type)
    ax.set_title(config.TASKS[task]['description'])
    ax.set_xlabel('Window')
    ax.set_ylabel('AUROC')
    ax.legend()
    ax.set_xticks([1,2,3,4,5])
plt.suptitle('Model Performance Degradation Across Temporal Windows', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/fig03_auroc_degradation.png', dpi=300, bbox_inches='tight')
plt.show()


## 4. Save Models and Results

In [ ]:
# Save all trained models and scaler
checkpoint_02 = {
    'models': models,
    'scaler': scaler,
    'performance': perf_df,
    'feature_cols': feature_cols,
}
with open('../outputs/checkpoint_02.pkl', 'wb') as f:
    pickle.dump(checkpoint_02, f)

# Also save performance table as CSV
perf_df.to_csv('../outputs/metrics/model_performance.csv', index=False)
print('Saved: checkpoint_02.pkl + model_performance.csv')
print(f'Models trained: {len(models)}')
